# EAGLE-3 Draft Training for Gemma 4 E2B (Colab end-to-end)

Trains an EAGLE-3 style draft model for speculative decoding on CoreML ANE.

Differences vs EAGLE-1 (NeurIPS 2025 paper):
- **Direct token prediction** (CE on next token), not feature (MSE) prediction.
- **Multi-layer feature fusion**: concat (low, mid, high) target hidden states.
- **Training-time test (TTT)**: simulate inference distribution by running the draft K steps autoregressively on its own hidden state and summing losses. This fixes the distribution mismatch that caps EAGLE-1 at ~50-60% acceptance.
- **Draft = 1 Gemma4-style decoder layer** (GQA + RMSNorm + SwiGLU + dual RoPE), not an FFN block.
- **Fixed tree K=3** for v1 (dynamic tree is v2).

**Inputs**: `eagle_corpus.jsonl` produced by `download_eagle_corpus.py --num-samples 30000` (already on Drive).
**Outputs to Drive**: `eagle3_draft_best.pt`, `eagle3_draft_final.pt`, `eagle3_config.json`, `eagle3_training.log`.

**Runtime**: A100 ≈ 2-4h for 2 epochs × 30k seqs × seq_len=512. T4 ≈ 10-14h.

Hidden states are collected **on-the-fly** (target model stays loaded). No 100GB+ disk dump.

## 0. Install + HF login + Drive mount

In [ ]:
!pip install -q -U transformers accelerate datasets safetensors huggingface_hub hf_transfer tqdm

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Paste your HF token here (https://huggingface.co/settings/tokens)
HF_TOKEN = ""  # e.g. "hf_abc..."
assert HF_TOKEN, "Paste your HF token above"
from huggingface_hub import login
login(token=HF_TOKEN)

from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

## 1. Config — edit these paths / hyperparams

In [ ]:
# ── Paths ──────────────────────────────────────────────────
CORPUS_PATH = "/content/drive/MyDrive/eagle_corpus.jsonl"
SAVE_DIR    = "/content/drive/MyDrive/eagle3_draft"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Model ──────────────────────────────────────────────────
MODEL_ID = "google/gemma-4-E2B-it"

# ── Data ───────────────────────────────────────────────────
NUM_SAMPLES = 30000        # sequences from corpus
SEQ_LEN     = 512          # tokens per seq
VAL_SAMPLES = 512          # held-out sequences for eval

# ── Draft architecture ─────────────────────────────────────
FUSION_LAYERS = [8, 17, 34]  # target layer indices for low / mid / high fusion (35-layer Gemma4-E2B)
INIT_FROM_TARGET_LAYER = 13  # warm-start from a K/V-owning layer (L15-34 are KV-shared and lack k_proj/v_proj). Set None to skip.

# ── Training ───────────────────────────────────────────────
EPOCHS          = 2
MICRO_BATCH     = 1                # sequences per forward (target is big; 1 is safest)
GRAD_ACCUM      = 8                # effective batch = MICRO_BATCH × GRAD_ACCUM
LR              = 3e-4
WEIGHT_DECAY    = 0.01
WARMUP_STEPS    = 500
GRAD_CLIP       = 1.0

# EAGLE-3 Training-Time Test
TTT_K           = 3                # number of autoregressive draft steps during training (matches inference K)
TTT_WEIGHTS     = [1.0, 0.7, 0.5]  # per-step loss weights (step 0..K-1); len must be TTT_K
FEATURE_LOSS_W  = 0.1              # auxiliary MSE(draft_hidden, target_hidden) weight (0 disables)

# ── Eval ───────────────────────────────────────────────────
EVAL_EVERY_STEPS = 2000
SAVE_EVERY_STEPS = 4000

# ── Seed / device ──────────────────────────────────────────
SEED   = 42
DEVICE = "cuda"

import torch, random, numpy as np
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name()}")
print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load target model (frozen) + read config

In [ ]:
from transformers import AutoTokenizer

try:
    from transformers import Gemma4ForConditionalGeneration as TargetCls
except Exception:
    # Fallback for older transformers
    from transformers import AutoModelForCausalLM as TargetCls

print(f"Loading {MODEL_ID} (fp16)...")
target = TargetCls.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map=DEVICE)
target.eval()
for p in target.parameters(): p.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Read actual config (avoid hardcoding)
tcfg = target.config.text_config if hasattr(target.config, "text_config") else target.config
HIDDEN      = tcfg.hidden_size
NUM_LAYERS  = tcfg.num_hidden_layers
NUM_HEADS   = tcfg.num_attention_heads
NUM_KV      = getattr(tcfg, "num_key_value_heads", NUM_HEADS)
HEAD_DIM    = getattr(tcfg, "head_dim", HIDDEN // NUM_HEADS)
FFN_DIM     = getattr(tcfg, "intermediate_size", HIDDEN * 4)
VOCAB       = tcfg.vocab_size
EMBED_SCALE = HIDDEN ** 0.5
RMS_EPS     = getattr(tcfg, "rms_norm_eps", 1e-6)
# Gemma4 has dual RoPE theta; for the draft we use the sliding (short) one
ROPE_THETA  = getattr(tcfg, "rope_local_base_freq", getattr(tcfg, "rope_theta", 10000.0))

print(f"hidden={HIDDEN} layers={NUM_LAYERS} heads={NUM_HEADS} kv={NUM_KV} head_dim={HEAD_DIM} ffn={FFN_DIM}")
print(f"vocab={VOCAB} embed_scale={EMBED_SCALE:.3f} rms_eps={RMS_EPS} rope_theta={ROPE_THETA}")

# Sanity check layer indices
assert all(0 <= i < NUM_LAYERS for i in FUSION_LAYERS), f"FUSION_LAYERS={FUSION_LAYERS} out of range for {NUM_LAYERS} layers"
print(f"Fusion picks target layers: {FUSION_LAYERS}")

## 3. Draft model (EAGLE-3): FeatureFusion + 1 Gemma4-style decoder layer

Input per position t:
- Either `FeatureFusion(h_low[t], h_mid[t], h_high[t])` (from target), or
- The draft's own previous hidden (autoregressive TTT / inference mode).

Then concat with `embed(next_token)`, project to HIDDEN, run through one decoder layer, emit logits via shared (frozen) lm_head.

Decoder layer mirrors Gemma4 target: GQA + Q-norm + K-norm + V-norm (weight-free, `with_scale=False`) + SwiGLU + RoPE. Matches target attention math exactly so warm-start from target layer 13 is clean.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        dtype = x.dtype
        x = x.float()
        n = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * n).to(dtype) * self.weight


class RMSNormNoScale(nn.Module):
    """RMS normalization without learnable scale (matches Gemma4 v_norm `with_scale=False`)."""
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    def forward(self, x):
        dtype = x.dtype
        x = x.float()
        n = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * n).to(dtype)


def build_rope_cache(max_seq, head_dim, theta, device, dtype=torch.float32):
    half = head_dim // 2
    inv_freq = 1.0 / (theta ** (torch.arange(0, half, device=device, dtype=dtype) / half))
    pos = torch.arange(max_seq, device=device, dtype=dtype)
    freqs = torch.einsum("i,j->ij", pos, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class FeatureFusion(nn.Module):
    def __init__(self, hidden, n_layers):
        super().__init__()
        self.proj = nn.Linear(hidden * n_layers, hidden, bias=False)
        self.norm = RMSNorm(hidden, eps=RMS_EPS)
    def forward(self, layer_hiddens):
        return self.norm(self.proj(torch.cat(layer_hiddens, dim=-1)))


class DraftDecoderLayer(nn.Module):
    """Single Gemma4-style decoder layer (GQA + QK-norm + SwiGLU + RoPE)."""
    def __init__(self):
        super().__init__()
        self.pre_attn_norm  = RMSNorm(HIDDEN, RMS_EPS)
        self.q_proj = nn.Linear(HIDDEN, NUM_HEADS * HEAD_DIM, bias=False)
        self.k_proj = nn.Linear(HIDDEN, NUM_KV    * HEAD_DIM, bias=False)
        self.v_proj = nn.Linear(HIDDEN, NUM_KV    * HEAD_DIM, bias=False)
        self.q_norm = RMSNorm(HEAD_DIM, RMS_EPS)
        self.k_norm = RMSNorm(HEAD_DIM, RMS_EPS)
        self.v_norm = RMSNormNoScale(RMS_EPS)  # matches Gemma4 target (with_scale=False)
        self.o_proj = nn.Linear(NUM_HEADS * HEAD_DIM, HIDDEN, bias=False)
        self.pre_ffn_norm   = RMSNorm(HIDDEN, RMS_EPS)
        self.gate_proj = nn.Linear(HIDDEN, FFN_DIM, bias=False)
        self.up_proj   = nn.Linear(HIDDEN, FFN_DIM, bias=False)
        self.down_proj = nn.Linear(FFN_DIM, HIDDEN, bias=False)

    def forward(self, x, cos, sin, causal=True):
        B, T, _ = x.shape
        h = self.pre_attn_norm(x)
        q = self.q_proj(h).view(B, T, NUM_HEADS, HEAD_DIM).transpose(1, 2)
        k = self.k_proj(h).view(B, T, NUM_KV,    HEAD_DIM).transpose(1, 2)
        v = self.v_proj(h).view(B, T, NUM_KV,    HEAD_DIM).transpose(1, 2)
        q = self.q_norm(q); k = self.k_norm(k); v = self.v_norm(v)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])
        rep = NUM_HEADS // NUM_KV
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        attn = attn.transpose(1, 2).contiguous().view(B, T, NUM_HEADS * HEAD_DIM)
        x = x + self.o_proj(attn)
        h = self.pre_ffn_norm(x)
        x = x + self.down_proj(F.silu(self.gate_proj(h)) * self.up_proj(h))
        return x


class EAGLE3Draft(nn.Module):
    def __init__(self, lm_head_weight):
        super().__init__()
        self.fusion     = FeatureFusion(HIDDEN, len(FUSION_LAYERS))
        self.input_proj = nn.Linear(HIDDEN * 2, HIDDEN, bias=False)
        self.layer      = DraftDecoderLayer()
        self.final_norm = RMSNorm(HIDDEN, RMS_EPS)
        self.register_buffer("lm_head_weight", lm_head_weight, persistent=False)

    def step(self, h_prev, e_next, cos, sin, is_sequence=True):
        x = torch.cat([h_prev, e_next], dim=-1)
        x = self.input_proj(x)
        x = self.layer(x, cos, sin, causal=is_sequence)
        x = self.final_norm(x)
        logits = F.linear(x.float(), self.lm_head_weight.float())
        return x, logits

    def fuse_target(self, layer_hiddens):
        return self.fusion(layer_hiddens)


def init_draft_from_target_layer(draft, target, layer_idx):
    """Best-effort warm-start. Silently skips mismatches — Gemma4 HF attention may use
    fused qkv_proj / kv_proj depending on transformers version."""
    if layer_idx is None: return
    try:
        tm = target.model
        if hasattr(tm, "language_model"): tm = tm.language_model
        if hasattr(tm, "model"): tm = tm.model
        tl = tm.layers[layer_idx]
    except Exception as e:
        print(f"  warm-start skipped: can't access target layer {layer_idx} ({e})")
        return

    attn = getattr(tl, "self_attn", None)
    mlp  = getattr(tl, "mlp", None)
    if attn is None or mlp is None:
        print(f"  warm-start skipped: no self_attn/mlp on layer {layer_idx}")
        return

    def tensor_at(mod, *names):
        for n in names:
            m = getattr(mod, n, None)
            if m is not None and hasattr(m, "weight"): return m.weight
        return None

    def copy_(dst, src):
        if src is None or dst.shape != src.shape: return False
        with torch.no_grad(): dst.copy_(src.detach().float())
        return True

    qkv = tensor_at(attn, "qkv_proj", "wqkv")
    kv  = tensor_at(attn, "kv_proj",  "wkv")
    q_w = tensor_at(attn, "q_proj", "wq", "query", "query_proj")
    k_w = tensor_at(attn, "k_proj", "wk", "key",   "key_proj")
    v_w = tensor_at(attn, "v_proj", "wv", "value", "value_proj")
    o_w = tensor_at(attn, "o_proj", "wo", "out_proj", "output_proj")

    if qkv is not None and q_w is None:
        qd = NUM_HEADS * HEAD_DIM; kd = NUM_KV * HEAD_DIM
        if qkv.shape[0] == qd + 2 * kd:
            q_w = qkv[:qd]; k_w = qkv[qd:qd + kd]; v_w = qkv[qd + kd:]
    if kv is not None and k_w is None:
        kd = NUM_KV * HEAD_DIM
        if kv.shape[0] == 2 * kd:
            k_w = kv[:kd]; v_w = kv[kd:]

    ok = 0
    if copy_(draft.layer.q_proj.weight, q_w): ok += 1
    if copy_(draft.layer.k_proj.weight, k_w): ok += 1
    if copy_(draft.layer.v_proj.weight, v_w): ok += 1
    if copy_(draft.layer.o_proj.weight, o_w): ok += 1
    if copy_(draft.layer.q_norm.weight, tensor_at(attn, "q_norm")): ok += 1
    if copy_(draft.layer.k_norm.weight, tensor_at(attn, "k_norm")): ok += 1
    if copy_(draft.layer.gate_proj.weight, tensor_at(mlp, "gate_proj", "w1", "gate")): ok += 1
    if copy_(draft.layer.up_proj.weight,   tensor_at(mlp, "up_proj",   "w3", "up"  )): ok += 1
    if copy_(draft.layer.down_proj.weight, tensor_at(mlp, "down_proj", "w2", "down")): ok += 1

    print(f"  warm-start: copied {ok} tensors from target layer {layer_idx}")
    if ok == 0:
        attn_names = [n for n, _ in attn.named_children()]
        mlp_names  = [n for n, _ in mlp.named_children()]
        print(f"    attn children: {attn_names}")
        print(f"    mlp  children: {mlp_names}")


# Build draft
lm_head_weight = target.lm_head.weight.data.detach().clone()
draft = EAGLE3Draft(lm_head_weight.float()).to(DEVICE)
init_draft_from_target_layer(draft, target, INIT_FROM_TARGET_LAYER)
n_params = sum(p.numel() for p in draft.parameters() if p.requires_grad)
print(f"Draft params: {n_params/1e6:.1f}M ({n_params/2.7e9*100:.2f}% of 2.7B target)")

COS, SIN = build_rope_cache(SEQ_LEN + 16, HEAD_DIM, ROPE_THETA, DEVICE)
print(f"RoPE cache: {COS.shape}")


## 4. Corpus loader + on-the-fly target forward

Returns per-sequence:
- `ids`:         (T,) int — token IDs from Gemma tokenizer
- `target_tok`:  (T-1,) int — what target predicts at each position (argmax of logits[t] ⇒ tok[t+1])
- `layer_h`:     list[(T, H)] fp16 — hidden states at FUSION_LAYERS
- `embeds`:      (T, H) fp16 — input embeddings × scale

In [ ]:
import json, time
from tqdm.auto import tqdm

print(f"Reading corpus {CORPUS_PATH}...")
texts = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        texts.append(json.loads(line)["text"])
print(f"  loaded {len(texts)} sequences")

random.Random(SEED).shuffle(texts)
val_texts   = texts[:VAL_SAMPLES]
train_texts = texts[VAL_SAMPLES:VAL_SAMPLES + NUM_SAMPLES]
print(f"  train: {len(train_texts)}  val: {len(val_texts)}")

EMBED_FN = target.get_input_embeddings()


@torch.no_grad()
def target_forward(text):
    """Tokenize, run target, return tensors. Multi-layer via output_hidden_states."""
    ids = tokenizer.encode(text, return_tensors="pt", truncation=True, max_length=SEQ_LEN).to(DEVICE)
    if ids.shape[1] < 32:
        return None
    out = target.model(input_ids=ids, output_hidden_states=True, use_cache=False)
    # HF returns (embed_output, layer_1_out, ..., layer_N_out) ⇒ index i+1 for layer i
    all_h = out.hidden_states
    layer_h = [all_h[i + 1][0].detach().half() for i in FUSION_LAYERS]    # list[(T, H)]
    last_h  = all_h[-1][0].detach()                                         # (T, H) fp16
    # target token labels: argmax of lm_head @ last_hidden at each position
    logits = F.linear(last_h.float(), target.lm_head.weight.float())       # (T, V)
    tok_tgt = logits.argmax(dim=-1).detach()                                # (T,)
    embeds = EMBED_FN(ids)[0].detach().half() * EMBED_SCALE                 # (T, H)
    return {
        "ids":        ids[0].detach().cpu(),
        "layer_h":    [h.detach() for h in layer_h],     # on GPU fp16
        "last_h":     last_h.detach(),                    # on GPU fp16
        "embeds":     embeds.detach(),                    # on GPU fp16
        "tok_tgt":    tok_tgt,                            # on GPU int64
    }

# Smoke test
smp = target_forward(train_texts[0])
print("smoke:", {k: (v.shape if hasattr(v,'shape') else [x.shape for x in v]) for k,v in smp.items()})

## 5. EAGLE-3 training step — Teacher-forced + Training-Time Test

Step 0 (TF): draft input = FeatureFusion(target layer hiddens at t) + embed(tok[t+1]), predict tok[t+1]. Label = target_tok[t].
Steps 1..K-1 (TTT): draft input = **draft's own previous hidden** + embed(draft's previous predicted token), predict tok shifted +k. Label = target_tok[t+k].

This matches exactly the inference-time recurrence and is the EAGLE-3 quality booster.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

assert len(TTT_WEIGHTS) == TTT_K, "TTT_WEIGHTS length must equal TTT_K"

opt = AdamW(draft.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
total_steps = (len(train_texts) // MICRO_BATCH) * EPOCHS
def lr_lambda(step):
    if step < WARMUP_STEPS: return step / max(1, WARMUP_STEPS)
    p = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * p))
import math
sched = LambdaLR(opt, lr_lambda)

scaler = torch.cuda.amp.GradScaler()


def train_step(batch_texts):
    """Processes MICRO_BATCH sequences end-to-end. Returns (loss_total, loss_step0, acc_step0, pairs)."""
    draft.train()
    loss_total = 0.0; loss_s0 = 0.0; correct_s0 = 0; n_pairs = 0
    for text in batch_texts:
        smp = target_forward(text)
        if smp is None: continue
        T = smp["last_h"].shape[0]
        if T < TTT_K + 2: continue
        # Positions t available for TTT: 0 .. T-1-TTT_K
        valid = T - TTT_K
        with torch.cuda.amp.autocast(dtype=torch.float16):
            # ── Step 0 (teacher-forced from target multi-layer fusion) ──
            h_prev = draft.fuse_target([h[:valid] for h in smp["layer_h"]]).unsqueeze(0)    # (1, valid, H)
            # e_in[t] = embed(tok[t+1])  (shift by 1)
            e_in   = smp["embeds"][1:valid + 1].unsqueeze(0)
            label0 = smp["tok_tgt"][:valid].unsqueeze(0)    # predicts token at t+1
            d_h, logits = draft.step(h_prev, e_in, COS, SIN, is_sequence=True)
            ce0 = F.cross_entropy(logits.view(-1, VOCAB), label0.view(-1))
            if FEATURE_LOSS_W > 0:
                mse = F.mse_loss(d_h.float(), smp["last_h"][1:valid+1].unsqueeze(0).float())
                loss = TTT_WEIGHTS[0] * ce0 + FEATURE_LOSS_W * mse
            else:
                loss = TTT_WEIGHTS[0] * ce0

            loss_s0 += ce0.item(); n_pairs += valid
            correct_s0 += (logits.argmax(-1) == label0).sum().item()

            # ── Steps 1..K-1 (TTT: draft-on-draft) ──
            for k in range(1, TTT_K):
                # Use draft's own predicted token embedding as e_in; draft's own hidden as h_prev.
                pred_tok = logits.argmax(-1)             # (1, valid)
                e_k = EMBED_FN(pred_tok).to(d_h.dtype) * EMBED_SCALE
                d_h, logits = draft.step(d_h, e_k, COS, SIN, is_sequence=True)
                label_k = smp["tok_tgt"][k:k + valid].unsqueeze(0)
                ce_k = F.cross_entropy(logits.view(-1, VOCAB), label_k.view(-1))
                loss = loss + TTT_WEIGHTS[k] * ce_k

        loss_total += loss.item()
        scaler.scale(loss / max(1, GRAD_ACCUM)).backward()

    return loss_total, loss_s0, correct_s0, n_pairs

## 6. Validation — K-step acceptance estimate

In [ ]:
@torch.no_grad()
def evaluate(n=None):
    draft.eval()
    n = n or len(val_texts)
    totals   = [0] * TTT_K
    correct  = [0] * TTT_K
    for text in val_texts[:n]:
        smp = target_forward(text)
        if smp is None: continue
        T = smp["last_h"].shape[0]
        if T < TTT_K + 2: continue
        valid = T - TTT_K
        with torch.cuda.amp.autocast(dtype=torch.float16):
            h_prev = draft.fuse_target([h[:valid] for h in smp["layer_h"]]).unsqueeze(0)
            e_in   = smp["embeds"][1:valid + 1].unsqueeze(0)
            d_h, logits = draft.step(h_prev, e_in, COS, SIN, is_sequence=True)
            label0 = smp["tok_tgt"][:valid].unsqueeze(0)
            correct[0] += (logits.argmax(-1) == label0).sum().item(); totals[0] += valid
            for k in range(1, TTT_K):
                pred_tok = logits.argmax(-1)
                e_k = EMBED_FN(pred_tok).to(d_h.dtype) * EMBED_SCALE
                d_h, logits = draft.step(d_h, e_k, COS, SIN, is_sequence=True)
                label_k = smp["tok_tgt"][k:k + valid].unsqueeze(0)
                correct[k] += (logits.argmax(-1) == label_k).sum().item(); totals[k] += valid
    accs = [c / max(1, t) for c, t in zip(correct, totals)]
    # Mean accepted length (simple greedy accept under exact-match; real accept uses sampling)
    # p(accept all up to step k) = prod accs[:k+1]
    cum = 1.0
    exp_len = 1.0  # the target verification always emits 1 token
    for a in accs:
        cum *= a
        exp_len += cum
    return accs, exp_len

## 7. Training loop with periodic eval + checkpointing to Drive

In [ ]:
import time
from pathlib import Path

log_f = open(f"{SAVE_DIR}/eagle3_training.log", "w")
def log(msg):
    print(msg)
    log_f.write(msg + "\n"); log_f.flush()

def save_checkpoint(tag, extra=None):
    p = f"{SAVE_DIR}/eagle3_draft_{tag}.pt"
    state = {"model": draft.state_dict(), "meta": {
        "hidden": HIDDEN, "num_heads": NUM_HEADS, "num_kv": NUM_KV, "head_dim": HEAD_DIM,
        "ffn": FFN_DIM, "vocab": VOCAB, "rms_eps": RMS_EPS, "rope_theta": ROPE_THETA,
        "embed_scale": EMBED_SCALE, "fusion_layers": FUSION_LAYERS, "ttt_k": TTT_K,
        "ttt_weights": TTT_WEIGHTS,
        "model_id": MODEL_ID,
    }}
    if extra: state["meta"].update(extra)
    torch.save(state, p)
    size_mb = Path(p).stat().st_size / 1e6
    log(f"  checkpoint: {p} ({size_mb:.1f} MB)")
    return p

# Save architecture config alongside for CoreML conversion
with open(f"{SAVE_DIR}/eagle3_config.json", "w") as f:
    json.dump({
        "architecture": "eagle3_draft",
        "hidden": HIDDEN, "num_heads": NUM_HEADS, "num_kv_heads": NUM_KV,
        "head_dim": HEAD_DIM, "ffn": FFN_DIM, "vocab": VOCAB, "rms_eps": RMS_EPS,
        "rope_theta": ROPE_THETA, "embed_scale": EMBED_SCALE,
        "fusion_layers": FUSION_LAYERS, "ttt_k": TTT_K, "model_id": MODEL_ID,
    }, f, indent=2)

best_acc0 = 0.0
step = 0; seen = 0
t_epoch0 = time.time()

for epoch in range(EPOCHS):
    log(f"\n=== Epoch {epoch + 1}/{EPOCHS} ===")
    random.Random(SEED + epoch).shuffle(train_texts)
    opt.zero_grad()
    pbar = tqdm(range(0, len(train_texts), MICRO_BATCH), desc=f"epoch {epoch+1}")
    loss_ema = None; acc_ema = None
    for i in pbar:
        batch = train_texts[i:i + MICRO_BATCH]
        loss, loss0, corr0, pairs = train_step(batch)
        seen += 1
        if (seen % GRAD_ACCUM) == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(draft.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
            step += 1
        if pairs > 0:
            a0 = corr0 / pairs
            loss_ema = loss0 if loss_ema is None else 0.98 * loss_ema + 0.02 * loss0
            acc_ema  = a0    if acc_ema  is None else 0.98 * acc_ema  + 0.02 * a0
            pbar.set_postfix({"ce0": f"{loss_ema:.3f}", "acc0": f"{acc_ema*100:.1f}%",
                              "lr": f"{sched.get_last_lr()[0]:.2e}"})

        if step > 0 and (step % EVAL_EVERY_STEPS) == 0 and (seen % GRAD_ACCUM) == 0:
            accs, exp_len = evaluate(n=128)
            log(f"  [eval @ step {step}] accs={[f'{a*100:.1f}' for a in accs]} expL={exp_len:.2f}")
            if accs[0] > best_acc0:
                best_acc0 = accs[0]
                save_checkpoint("best", extra={"val_accs": accs, "val_exp_len": exp_len, "step": step})

        if step > 0 and (step % SAVE_EVERY_STEPS) == 0 and (seen % GRAD_ACCUM) == 0:
            save_checkpoint(f"step{step}")

log(f"\nTraining done in {(time.time() - t_epoch0) / 60:.1f} min.")
save_checkpoint("final")
log_f.close()

## 8. Final validation on held-out set

In [ ]:
accs, exp_len = evaluate()
print(f"Validation (full held-out = {len(val_texts)} seqs):")
for k, a in enumerate(accs):
    print(f"  step {k}: top-1 accuracy vs target = {a*100:.2f}%")
print(f"  expected accept length (greedy): {exp_len:.3f} tokens / forward")
print(f"  est. speedup (WFA @ 30 tok/s base): {exp_len:.2f}x = {30 * exp_len:.0f} tok/s")

with open(f"{SAVE_DIR}/eagle3_eval.json", "w") as f:
    json.dump({"accs": accs, "expected_length": exp_len,
               "base_tokps": 30, "est_tokps": 30 * exp_len}, f, indent=2)
print(f"\nEval saved to {SAVE_DIR}/eagle3_eval.json")

## 9. Artifacts on Drive

```
/content/drive/MyDrive/eagle3_draft/
  ├── eagle3_draft_best.pt    ← highest val acc[0]
  ├── eagle3_draft_final.pt   ← last epoch
  ├── eagle3_draft_stepNNNN.pt (periodic)
  ├── eagle3_config.json      ← architecture for CoreML conversion
  ├── eagle3_eval.json        ← final accuracy numbers
  └── eagle3_training.log
```

### Next step (not in this notebook)
1. Download `eagle3_draft_best.pt` + `eagle3_config.json` to the Mac.
2. Extend `build_speculative.py` with an `--eagle3-path` mode that:
   - Loads the draft (1 Gemma4-style decoder layer + fusion + input_proj).
   - Converts it as `eagle3.mlpackage` using the existing ane_ops.
   - Adds **multi-layer hidden outputs** to target chunks at `FUSION_LAYERS` → new `chunk{1,2,3,4}_with_mid.mlpackage`.
   - Builds `verify_chunk{1..4}_K3.mlpackage` with `seq_dim=3` using EnumeratedShapes.
3. Swift verify loop with fixed K=3 (dynamic tree is v2).

### Notes
- If acc[0] plateaus below ~55%, increase `NUM_SAMPLES` (corpus already has 30k but you can regenerate with `--num-samples 60000`) or train for more epochs.
- If acc[1]/acc[2] drop sharply vs acc[0], TTT weights or FEATURE_LOSS_W are off — raise TTT weights for later steps.
- `INIT_FROM_TARGET_LAYER` warm-start typically gives +5-10pt acc[0] early in training.
- Memory tip: if OOM on T4, drop `SEQ_LEN` to 384 and `GRAD_ACCUM` to 4.